# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # Load variables from .env file
print("PRICE_DATA path:", os.getenv("PRICE_DATA"))


PRICE_DATA path: ../../05_src/data/prices/


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [4]:
import dask.dataframe as dd

price_dd = dd.read_parquet(os.path.join(price_data_path, "**"), engine="pyarrow")
print(price_dd.head())


        Date       Open       High        Low      Close  Adj Close  Volume  \
0 2011-01-27  18.980698  19.028151  18.942738  18.980698   7.294248  353500   
1 2011-01-28  19.009169  19.161015  19.009169  19.075602   7.330718   16500   
2 2011-01-31  19.161015  19.293880  19.161015  19.170506   7.367188    7700   
3 2011-02-01  19.170506  19.170506  18.990189  19.075602   7.330718   10700   
4 2011-02-02  19.028151  19.047131  18.980698  19.028151   7.312487   12700   

    source ticker  Year  
0  ACP.csv    ACP  2011  
1  ACP.csv    ACP  2011  
2  ACP.csv    ACP  2011  
3  ACP.csv    ACP  2011  
4  ACP.csv    ACP  2011  


In [5]:
from dotenv import load_dotenv
import os
from glob import glob

# Load environment variables
load_dotenv()

# Load PRICE_DATA path
price_data_path = os.getenv("PRICE_DATA")

# Use glob to find all parquet part files in subdirectories
parquet_files = glob(os.path.join(price_data_path, "**", "*.parquet"), recursive=True)

# Print result
print(f"Found {len(parquet_files)} parquet files:")
for path in parquet_files[:5]:  # show a few as example
    print(path)


Found 2646 parquet files:
../../05_src/data/prices/DENN/DENN_1998/part.1.parquet
../../05_src/data/prices/DENN/DENN_1998/part.0.parquet
../../05_src/data/prices/DENN/DENN_2010/part.1.parquet
../../05_src/data/prices/DENN/DENN_2010/part.0.parquet
../../05_src/data/prices/DENN/DENN_1999/part.1.parquet


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [6]:
# Write your code below.
import dask.dataframe as dd
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
price_data_path = os.getenv("PRICE_DATA")

# Load all parquet files as one Dask dataframe
dd_prices = dd.read_parquet(os.path.join(price_data_path, "**"), engine="pyarrow")

# Make sure Date is datetime
dd_prices['Date'] = dd.to_datetime(dd_prices['Date'])

# Sort and compute features per ticker
dd_feat = (dd_prices
    .set_index("Date")
    .groupby("ticker")
    .apply(lambda df: df.assign(
        Close_lag_1 = df["Close"].shift(1),
        Adj_Close_lag_1 = df["Adj Close"].shift(1),
        returns = (df["Close"] / df["Close"].shift(1)) - 1,
        hi_lo_range = df["High"] - df["Low"]
    ), meta={
        'Open': 'f8',
        'High': 'f8',
        'Low': 'f8',
        'Close': 'f8',
        'Adj Close': 'f8',
        'Volume': 'i8',
        'source': 'object',
        'ticker': 'object',
        'Year': 'i8',
        'Close_lag_1': 'f8',
        'Adj_Close_lag_1': 'f8',
        'returns': 'f8',
        'hi_lo_range': 'f8'
    })
)

# Optional: preview the result
dd_feat.head()



/home/codespace/.python/current/lib/python3.12/site-packages/dask/dataframe/core.py:382: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range


In [ ]:
# Confirm data is loaded
print(dd_prices.shape)
print(dd_prices.head())


(<dask_expr.expr.Scalar: expr=Assign(frame=ReadParquetFSSpec(4f2040b)).size() // 10, dtype=int64>, 10)
        Date       Open       High        Low      Close  Adj Close  Volume  \
0 2011-01-27  18.980698  19.028151  18.942738  18.980698   7.294248  353500   
1 2011-01-28  19.009169  19.161015  19.009169  19.075602   7.330718   16500   
2 2011-01-31  19.161015  19.293880  19.161015  19.170506   7.367188    7700   
3 2011-02-01  19.170506  19.170506  18.990189  19.075602   7.330718   10700   
4 2011-02-02  19.028151  19.047131  18.980698  19.028151   7.312487   12700   

    source ticker  Year  
0  ACP.csv    ACP  2011  
1  ACP.csv    ACP  2011  
2  ACP.csv    ACP  2011  
3  ACP.csv    ACP  2011  
4  ACP.csv    ACP  2011  


In [ ]:
# Confirm Date parsing
print(dd_prices.dtypes)


Date          datetime64[ns]
Open                 float64
High                 float64
Low                  float64
Close                float64
Adj Close            float64
Volume               float64
source       string[pyarrow]
ticker       string[pyarrow]
Year                 float64
dtype: object


In [ ]:
#Check If DataFrame Has Rows
print(dd_prices.shape)


(<dask_expr.expr.Scalar: expr=Assign(frame=ReadParquetFSSpec(4f2040b)).size() // 10, dtype=int64>, 10)


In [ ]:
print(dd_prices.head())  # shows first few rows


        Date       Open       High        Low      Close  Adj Close  Volume  \
0 2011-01-27  18.980698  19.028151  18.942738  18.980698   7.294248  353500   
1 2011-01-28  19.009169  19.161015  19.009169  19.075602   7.330718   16500   
2 2011-01-31  19.161015  19.293880  19.161015  19.170506   7.367188    7700   
3 2011-02-01  19.170506  19.170506  18.990189  19.075602   7.330718   10700   
4 2011-02-02  19.028151  19.047131  18.980698  19.028151   7.312487   12700   

    source ticker  Year  
0  ACP.csv    ACP  2011  
1  ACP.csv    ACP  2011  
2  ACP.csv    ACP  2011  
3  ACP.csv    ACP  2011  
4  ACP.csv    ACP  2011  


In [11]:
import dask.dataframe as dd
from dotenv import load_dotenv
import os

load_dotenv()
price_data_path = os.getenv("PRICE_DATA")

# Load parquet files using Dask (make sure path and files exist)
dd_prices = dd.read_parquet(os.path.join(price_data_path, "**"), engine="pyarrow")

# Convert 'Date' column to datetime
dd_prices['Date'] = dd.to_datetime(dd_prices['Date'])

# Compute features grouped by ticker
dd_feat = (dd_prices
    .set_index('Date')
    .groupby('ticker')
    .apply(lambda df: df.assign(
        Close_lag_1 = df['Close'].shift(1),
        Adj_Close_lag_1 = df['Adj Close'].shift(1),
        returns = (df['Close'] / df['Close'].shift(1)) - 1,
        hi_lo_range = df['High'] - df['Low']
    ), meta={
        'Open': 'f8', 'High': 'f8', 'Low': 'f8', 'Close': 'f8',
        'Adj Close': 'f8', 'Volume': 'i8', 'source': 'object',
        'ticker': 'object', 'Year': 'i8',
        'Close_lag_1': 'f8', 'Adj_Close_lag_1': 'f8',
        'returns': 'f8', 'hi_lo_range': 'f8'
    })
)

print(dd_feat.head(5))


Empty DataFrame
Columns: [Open, High, Low, Close, Adj Close, Volume, source, ticker, Year, Close_lag_1, Adj_Close_lag_1, returns, hi_lo_range]
Index: []


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

import dask.dataframe as dd
import os
from dotenv import load_dotenv

# Step 1: Load environment variable
load_dotenv()
price_data_path = os.getenv("PRICE_DATA")

# Step 2: Load data from parquet
dd_prices = dd.read_parquet(os.path.join(price_data_path, "**"), engine="pyarrow")

# Step 3: Ensure 'Date' column is datetime
dd_prices['Date'] = dd.to_datetime(dd_prices['Date'])

# Step 4: Create Dask features
dd_feat = (dd_prices
    .set_index("Date")
    .groupby("ticker")
    .apply(lambda df: df.assign(
        Close_lag_1 = df["Close"].shift(1),
        Adj_Close_lag_1 = df["Adj Close"].shift(1),
        returns = (df["Close"] / df["Close"].shift(1)) - 1,
        hi_lo_range = df["High"] - df["Low"]
    ), meta={
        'Open': 'f8', 'High': 'f8', 'Low': 'f8', 'Close': 'f8',
        'Adj Close': 'f8', 'Volume': 'i8', 'source': 'object',
        'ticker': 'object', 'Year': 'i8',
        'Close_lag_1': 'f8', 'Adj_Close_lag_1': 'f8',
        'returns': 'f8', 'hi_lo_range': 'f8'
    })
)

# Step 5: Convert Dask to pandas
df_feat = dd_feat.compute()

# Step 6: Sort and add 10-day moving average of returns
df_feat = df_feat.sort_values(['ticker', 'Date'])
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].transform(lambda x: x.rolling(10).mean())

# Step 7: Show preview
print(df_feat[['ticker', 'Date', 'returns', 'returns_ma_10']].head(10))








/home/codespace/.python/current/lib/python3.12/site-packages/dask/dataframe/core.py:382: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(
/home/codespace/.python/current/lib/python3.12/site-packages/dask/dataframe/core.py:382: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
Not always. But in this case, we used pandas because it was simple and easy for small data.
+ Would it have been better to do it in Dask? Why?
Yes, if the data is very big. Dask can handle large data that doesn’t fit in memory. It also makes the process faster by using multiple cores. But Dask code is a bit more complex than pandas.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.